# Stability Criteria: Weak-Layer Parameter Comparison

Compares the **13 weak-layer method combinations** for computing WEAC g_delta,
with the slab parameterization fixed to the highest-coverage pathway
(kim_jamieson_table2 → wautier → kochle).

| σ_c method | σ_comp method | density_wl method | Notes |
|---|---|---|---|
| weissgraeber_rosendahl | reiweger | — | constant, always available |
| weissgraeber_rosendahl | mellor | geldsetzer / kim_jamieson_table2 / kim_jamieson_table5 / data_flow | |
| sigrist | reiweger | geldsetzer / kim_jamieson_table2 / kim_jamieson_table5 / data_flow | |
| sigrist | mellor | geldsetzer / kim_jamieson_table2 / kim_jamieson_table5 / data_flow | |

See `stability_criteria_inputs.ipynb` for the 32 slab-pathway analysis
(fixed weak-layer defaults: weissgraeber_rosendahl + reiweger).


In [1]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import numpy as np
import pandas as pd
import plotly.graph_objects as go

from snowpyt_mechparams.graph import graph
from snowpyt_mechparams.algorithm import find_parameterizations
from snowpyt_mechparams.execution import ExecutionEngine
from snowpyt_mechparams.execution.config import ExecutionConfig
from notebook_utils import (
    load_pits, create_ectp_slabs,
    nominal, count_ok, extract_param_stats,
    DENSITY_COLORS, rgba,
)

try:
    from tqdm import tqdm
except ImportError:
    def tqdm(x, **kwargs): return x


In [2]:
pits = load_pits()
ectp_slabs = create_ectp_slabs(pits)
total_slabs = len(ectp_slabs)
print(f'Loaded {len(pits):,} pits → {total_slabs:,} ECTP slabs')


Loaded 50,278 pits → 14,776 ECTP slabs


In [3]:
# All g_delta pathways: 32 slab × 13 weak-layer = 416
weac_paths = find_parameterizations(graph, graph.get_node('g_delta'))
print(f'Total g_delta pathways: {len(weac_paths)}  (32 slab × 13 weak-layer combinations)')


def get_all_segments(path):
    segs = []
    for branch in path.branches:
        segs.extend(branch.segments)
    for _, _, continuation in path.merge_points:
        segs.extend(continuation)
    return segs


def has_slab_methods(path, density, emod, nu):
    segs = get_all_segments(path)
    return (
        any(s.to_node == 'density'         and s.edge_name == density for s in segs) and
        any(s.to_node == 'elastic_modulus' and s.edge_name == emod    for s in segs) and
        any(s.to_node == 'poissons_ratio'  and s.edge_name == nu      for s in segs)
    )


# Fix the slab to the highest-coverage slab pathway
# (kim_jamieson_table2 density → wautier E-mod → kochle ν)
FIXED_DENSITY = 'kim_jamieson_table2'
FIXED_EMOD    = 'wautier'
FIXED_NU      = 'kochle'

wl_paths = [p for p in weac_paths if has_slab_methods(p, FIXED_DENSITY, FIXED_EMOD, FIXED_NU)]
print(f'Weak-layer pathways (slab fixed to {FIXED_DENSITY} / {FIXED_EMOD} / {FIXED_NU}): {len(wl_paths)}')
print()
for i, p in enumerate(wl_paths, 1):
    print(f'  {i}: {p}')


Total g_delta pathways: 416  (32 slab × 13 weak-layer combinations)
Weak-layer pathways (slab fixed to kim_jamieson_table2 / wautier / kochle): 13

  1: branch 1: snow_pit -- data_flow --> measured_density -- data_flow --> density -- data_flow --> merge_weac_inputs
branch 2: snow_pit -- data_flow --> measured_density -- data_flow --> density -- data_flow --> merge_density_grain_form
branch 3: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_density_grain_form
branch 4: snow_pit -- data_flow --> measured_grain_form -- kochle --> poissons_ratio -- data_flow --> merge_weac_inputs
branch 5: snow_pit -- data_flow --> measured_hand_hardness -- data_flow --> merge_hand_hardness_grain_form
branch 6: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_hand_hardness_grain_form
branch 7: snow_pit -- data_flow --> measured_grain_form -- data_flow --> merge_density_grain_form
branch 8: snow_pit -- weissgraeber_rosendahl --> G_c -- data_flow --> merge_weac_inputs

In [ ]:
engine = ExecutionEngine(graph)
config = ExecutionConfig(include_method_uncertainty=False, weac_timeout_seconds=5.0)

wl_records = []

for slab in tqdm(ectp_slabs, desc='WEAC weak-layer paths', unit='slab'):
    results  = engine.execute_all(slab, 'g_delta', config=config, pathways=wl_paths)
    n_layers = len(slab.layers)

    for pw in results.pathways.values():
        methods = pw.methods_used

        sigma_c_m    = methods.get('sigma_c',            '?')
        sigma_comp_m = methods.get('sigma_comp',         '?')
        density_wl_m = methods.get('density_weak_layer', 'none')  # 'none' = constant path

        # Slab parameter success (all layers must have values)
        ok_density = count_ok(pw.computation_trace, 'density',         n_layers)
        ok_emod    = count_ok(pw.computation_trace, 'elastic_modulus', n_layers)
        ok_nu      = count_ok(pw.computation_trace, 'poissons_ratio',  n_layers)
        ok_G       = count_ok(pw.computation_trace, 'shear_modulus',   n_layers)
        slab_ok    = ok_density and ok_emod and ok_nu and ok_G

        # g_delta output
        weac_res = pw.slab.weac_result
        g_val = np.nan
        if weac_res is not None:
            g_val = float(weac_res.g_delta)
        g_ok = not np.isnan(g_val)

        wl_records.append({
            'slab_id':             slab.slab_id,
            'sigma_c_method':      sigma_c_m,
            'sigma_comp_method':   sigma_comp_m,
            'density_wl_method':   density_wl_m,
            'slab_ok':             slab_ok,
            'g_delta':             g_val,
            'g_delta_ok':          g_ok,
            'unstable_weac':       (g_val >= 1.0) if g_ok else None,
        })

wl_df = pd.DataFrame(wl_records)
print(f'Records: {len(wl_df):,}  ({len(wl_df) // total_slabs} per slab × {total_slabs:,} slabs)')

WEAC weak-layer paths:   4%|▍         | 631/14776 [07:20<1:57:56,  2.00slab/s]

In [ ]:
def wl_label(sigma_c, sigma_comp, density_wl):
    if sigma_c == 'weissgraeber_rosendahl' and sigma_comp == 'reiweger':
        return 'wr + reiweger (constant)'
    parts = []
    if sigma_c == 'sigrist':
        parts.append('σ_c=sigrist')
    if sigma_comp == 'mellor':
        parts.append('σ_comp=mellor')
    if density_wl not in ('none', '?'):
        parts.append(f'ρ_wl={density_wl}')
    return ' / '.join(parts) if parts else 'wr + reiweger (constant)'

wl_df['wl_label'] = wl_df.apply(
    lambda r: wl_label(r['sigma_c_method'], r['sigma_comp_method'], r['density_wl_method']),
    axis=1
)

cov = (
    wl_df
    .groupby(['sigma_c_method', 'sigma_comp_method', 'density_wl_method', 'wl_label'])
    .agg(
        n_slab_ok = ('slab_ok',    'sum'),
        n_g_ok    = ('g_delta_ok', 'sum'),
        g_median  = ('g_delta',    'median'),
        g_mean    = ('g_delta',    'mean'),
    )
    .reset_index()
    .sort_values('n_g_ok', ascending=False)
)

print('Weak-layer pathway comparison')
print(f'  Fixed slab: {FIXED_DENSITY} → {FIXED_EMOD} → {FIXED_NU}')
print(f'  Total slabs: {total_slabs:,}')
print()
print(f"  {'Weak-layer combination':<42}  {'Slab OK':>22}  {'g_delta OK':>22}  {'Median gδ':>10}")
print(f"  {'-'*102}")
for _, row in cov.iterrows():
    ns = int(row['n_slab_ok'])
    ng = int(row['n_g_ok'])
    gm = f"{row['g_median']:.3f}" if not np.isnan(row['g_median']) else '—'
    print(f"  {row['wl_label']:<42}  "
          f"{ns:>5} / {total_slabs} ({ns/total_slabs:.1%})  "
          f"{ng:>5} / {total_slabs} ({ng/total_slabs:.1%})  "
          f"{gm:>10}")


In [ ]:
# Horizontal bar chart: g_delta coverage per weak-layer combination
cov_sorted = cov.sort_values('n_g_ok')

fig = go.Figure()
fig.add_trace(go.Bar(
    name='g_delta computed',
    x=(cov_sorted['n_g_ok'] / total_slabs * 100).tolist(),
    y=cov_sorted['wl_label'].tolist(),
    orientation='h',
    marker_color=rgba('#0072B2', 0.80),
    text=[(f"{v:.1f}%") for v in (cov_sorted['n_g_ok'] / total_slabs * 100).tolist()],
    textposition='outside',
))
fig.add_trace(go.Bar(
    name='slab OK (no g_delta)',
    x=((cov_sorted['n_slab_ok'] - cov_sorted['n_g_ok']) / total_slabs * 100).tolist(),
    y=cov_sorted['wl_label'].tolist(),
    orientation='h',
    marker_color=rgba('#E69F00', 0.60),
))

fig.update_layout(
    title=dict(
        text=(
            '<b>WEAC g_delta: Weak-Layer Pathway Coverage</b><br>'
            f'<sup>Fixed slab: {FIXED_DENSITY} → {FIXED_EMOD} → {FIXED_NU} | '
            f'{total_slabs:,} ECTP slabs</sup>'
        ),
        x=0.5, xanchor='center', font=dict(size=13),
    ),
    barmode='stack',
    xaxis=dict(title='Coverage (%)', range=[0, 100], gridcolor='rgba(200,200,200,0.4)'),
    yaxis=dict(tickfont=dict(size=10)),
    plot_bgcolor='white',
    legend=dict(orientation='h', x=0.5, xanchor='center', y=-0.15),
    width=900, height=520,
    margin=dict(l=250, r=100, t=90, b=80),
)
fig.show()


In [ ]:
wl_df.to_parquet('weac_weak_layer_results.parquet', index=False, engine='pyarrow')
print(f'Saved weac_weak_layer_results.parquet ({len(wl_df):,} rows)')
